In [4]:
import pymupdf
from pathlib import Path
from pprint import pprint

DOC_DIR = fr"./sample_docx"
DOC_PATH = Path(DOC_DIR, "digital.pdf")

pprint(f"Reading -> {DOC_PATH}")

doc = pymupdf.open(DOC_PATH)
pprint(doc.page_count)
text = ""   
for page in doc:
    print(page)
    text += page.get_text()

print(text)

'Reading -> sample_docx/digital.pdf'
1
page 0 of sample_docx/digital.pdf
 
 
PDF Test File 
 
Congratulations, your computer is equipped with a PDF (Portable Document Format) 
reader!  You should be able to view any of the PDF documents and forms available on 
our site.  PDF forms are indicated by these icons: 
  or  
.   
 
Yukon Department of Education 
Box 2703 
Whitehorse,Yukon 
Canada 
Y1A 2C6 
 
Please visit our website at:  http://www.education.gov.yk.ca/
   



In [ ]:
import pymupdf, pytesseract, io
from pathlib import Path
from PIL import Image

DOC_PATH = Path("./sample_docx/Scanned-BG-1.pdf")

print(f"File exists: {DOC_PATH.exists()}")

doc = pymupdf.open(DOC_PATH)
print(f"Page count: {doc.page_count}")
text = ""
for i, page in enumerate(doc):
    pix = page.get_pixmap(dpi=300)
    img_bytes = pix.tobytes("png")
    img = Image.open(io.BytesIO(img_bytes))
    
    # Run Tesseract on the image
    text += f"\n{pytesseract.image_to_string(img)}\n"
print(text)

File exists: True
Page count: 1

 

Kena

Education
Education

PDF Test File

Congratulations, your computer is equipped with a PDF (Portable Document Format)
reader! You should be able to view any of the PDF documents and forms available on
our site. PDF forms are indicated by these icons: 4. or Td.

Yukon Department of Education
Box 2703

Whitehorse, Yukon

Canada

Y1A 2C6

Please visit our website at: http:/Awww.education.gov.yk.ca/




In [ ]:
import traceback
from PIL import Image
import pymupdf, pytesseract, io

def get_ocr(file_path:str="")->dict:
    """
    Function which will get the raw text from the document and return a structured response

    Args:
        file_path (str, optional): File path from which we want to extract the raw text. Defaults to "".

    Returns:
        dict: 
        structure : {
        "text": <Extracted text>,
        "method": <Module of extraction>,
        "page_count": <Total pages in the document>,
        "status": < 1 : success, -1 : error in processing , 0 : processed but no extraction>
    }
    """
    text = ""
    result = {
        "text": "",
        "method": "",
        "page_count": 0,
        "status": -1
    }
    if not file_path:
        return {
            "text": "",
            "method": "NA",
            "page_count": 0,
            "status": -1
        }
    try:
        # trying extraction from pymupdf
        doc = pymupdf.open(file_path)
        result['page_count'] = doc.page_count
        for page in doc:
            text += page.get_text()
        
        if text:
            # if text extracted from the pymupdf libarary 
            result['text'] = text
            result['method'] = 'pymupdf'
            result['status'] = 1
        else:
            # trying to extract text from tesseract
            for i, page in enumerate(doc):
                pix = page.get_pixmap(dpi=300)
                img_bytes = pix.tobytes("png")
                img = Image.open(io.BytesIO(img_bytes))
                
                # Run Tesseract on the image
                text += f"\n{pytesseract.image_to_string(img)}\n"

            if text:
                result['text'] = text
                result['method'] = 'tesseract'
                result['status'] = 1
            else:
                result['text'] = text
                result['method'] = 'tesseract'
                result['status'] = 0

    except Exception as e:
        result['status'] = -1
        result['error'] = fr"{str(e)}\n{traceback.format_exc()}"
    return result

In [14]:
DOC_DIR = fr"./sample_docx"
digital_pdf = Path(DOC_DIR, "digital.pdf")
scanned_pdf = Path(DOC_DIR, "Scanned-BG-1.pdf")
get_ocr(file_path=digital_pdf)

{'text': ' \n \nPDF Test File \n \nCongratulations, your computer is equipped with a PDF (Portable Document Format) \nreader!  You should be able to view any of the PDF documents and forms available on \nour site.  PDF forms are indicated by these icons: \n  or  \n.   \n \nYukon Department of Education \nBox 2703 \nWhitehorse,Yukon \nCanada \nY1A 2C6 \n \nPlease visit our website at:  http://www.education.gov.yk.ca/\n   \n',
 'method': 'pymupdf',
 'page_count': 1,
 'status': 1}